In [ ]:
import ee
import geemap
import pandas as pd
import os

# Authenticate and initialize Earth Engine
ee.Authenticate()
ee.Initialize(project='ee-musa650hw02')

# Mount Google Drive if using Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


2. ROI

In [ ]:
# Define point of interest (San Francisco in this example)
point = ee.Geometry.Point([-122.4194, 37.7749])

# Create a buffer with radius of 8km and bounds
buffer = point.buffer(8000)
region = buffer.bounds()

3. Image collection and preprocessing

In [ ]:
# Get Landsat 8 Collection 2 Level-2 imagery
image = (
    ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
    .filterBounds(point)
    .filterDate("2023-09-01", "2023-09-30")
    .sort("CLOUD_COVER")  # Sort by cloud cover to get clearest image
    .first()
    .select(["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"])
)

# Clip the image to our region of interest
clipped_image = image.clip(region)

# Visualize the natural color image
vis_params = {
    "min": 5000,
    "max": 15000,
    "bands": ["SR_B4", "SR_B3", "SR_B2"]
}

Map = geemap.Map()
Map.centerObject(point, 12)
Map.addLayer(clipped_image, vis_params, "Landsat-8 (RGB)")
Map

Map(center=[37.7749, -122.41939999999998], controls=(WidgetControl(options=['position', 'transparent_bg'], wid…

4. freature engineering

In [ ]:
# Calculate spectral indices
# NDVI (Normalized Difference Vegetation Index)
ndvi = clipped_image.normalizedDifference(["SR_B5", "SR_B4"]).rename("NDVI")

# NDBI (Normalized Difference Built-up Index)
ndbi = clipped_image.normalizedDifference(["SR_B6", "SR_B5"]).rename("NDBI")

# MNDWI (Modified Normalized Difference Water Index)
mndwi = clipped_image.normalizedDifference(["SR_B3", "SR_B6"]).rename("MNDWI")

# Get elevation data from SRTM
dem = ee.Image("USGS/SRTMGL1_003").clip(region).rename("elevation")

# Calculate slope from DEM
slope = ee.Terrain.slope(dem).rename("slope")

# Add features to the image
clipped_image = clipped_image.addBands([ndvi, ndbi, mndwi, dem, slope])

# Display the band names to verify
print("Available bands:", clipped_image.bandNames().getInfo())

# BONUS: Add edge detection using Sobel filters
sobel_kernel = ee.Kernel.sobel()

# Apply Sobel kernel to selected bands
sobel_b2 = clipped_image.select("SR_B2").convolve(sobel_kernel).rename("sobel_b2")
sobel_b3 = clipped_image.select("SR_B3").convolve(sobel_kernel).rename("sobel_b3")
sobel_b4 = clipped_image.select("SR_B4").convolve(sobel_kernel).rename("sobel_b4")
sobel_b5 = clipped_image.select("SR_B5").convolve(sobel_kernel).rename("sobel_b5")

# Add edge detection bands
clipped_image = clipped_image.addBands([sobel_b2, sobel_b3, sobel_b4, sobel_b5])

# Verify all bands
print("Final bands:", clipped_image.bandNames().getInfo())

Available bands: ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'NDVI', 'NDBI', 'MNDWI', 'elevation', 'slope']
Final bands: ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7', 'NDVI', 'NDBI', 'MNDWI', 'elevation', 'slope', 'sobel_b2', 'sobel_b3', 'sobel_b4', 'sobel_b5']


5. feature normalization

In [ ]:
# Normalize all bands to 0-1 scale
band_names = ["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7",
              "NDVI", "NDBI", "MNDWI", "elevation", "slope",
              "sobel_b2", "sobel_b3", "sobel_b4", "sobel_b5"]

def normalize_band(image, band):
    min_val = image.select(band).reduceRegion(
        reducer=ee.Reducer.min(),
        geometry=image.geometry(),
        scale=30,
        bestEffort=True
    ).getNumber(band)

    max_val = image.select(band).reduceRegion(
        reducer=ee.Reducer.max(),
        geometry=image.geometry(),
        scale=30,
        bestEffort=True
    ).getNumber(band)

    return image.expression(
        "(b - min) / (max - min)",
        {"b": image.select(band), "min": min_val, "max": max_val}
    ).rename(band + "_norm")

# Normalize all bands
normalized_bands = [normalize_band(clipped_image, band) for band in band_names]

# Combine normalized bands
normalized_image = ee.Image.cat(normalized_bands)

# Display normalized false color image
norm_vis_params = {
    "min": 0,
    "max": 1,
    "bands": ["SR_B5_norm", "SR_B4_norm", "SR_B3_norm"]  # NIR, Red, Green
}

Map = geemap.Map()
Map.centerObject(point, 12)
Map.addLayer(normalized_image, norm_vis_params, "Normalized Image")
Map

Map(center=[37.7749, -122.41939999999998], controls=(WidgetControl(options=['position', 'transparent_bg'], wid…

6. creating and loading training data

In [ ]:
# # Example of geemap interactive sampling
# # NOTE: This is a simplified example. In practice, you'd use Map.draw_features()
# # followed by exporting and importing the data

# Map = geemap.Map()
# Map.centerObject(point, 12)
# Map.addLayer(clipped_image, vis_params, "Landsat Image")

# # This is where you would interactively collect points
# # For example:
# # training_data = Map.draw_features(
# #     geometry_type="point",
# #     description="Draw training points"
# # )
# Map

In [ ]:
# Load training data from GEE asset
asset_id = "projects/ee-musa650hw02/assets/labeling"
label_samples = ee.FeatureCollection(asset_id)

# Display sample count and labels
print("Number of samples:", label_samples.size().getInfo())
print("Labels:", label_samples.aggregate_array("label").distinct().getInfo())

# Visualize training samples on map
Map = geemap.Map()
Map.centerObject(point, 12)
Map.addLayer(normalized_image, norm_vis_params, "Normalized Image")
Map.addLayer(label_samples, {}, "Training Samples")
Map

Number of samples: 722
Labels: []


Map(center=[37.7749, -122.41939999999998], controls=(WidgetControl(options=['position', 'transparent_bg'], wid…

7. train test

In [ ]:
# First, let's check what your label_samples contains
print("Label samples info:", label_samples.first().getInfo())

# Check if there are any features in the collection
print("Number of features:", label_samples.size().getInfo())

# If the above line also fails with the same error, your feature collection might be corrupted
# Let's try a direct approach to create new training data

# Define classes and colors for visualization (adjust based on your specific classes)
class_info = {
    0: {'name': 'Urban', 'color': 'red'},
    1: {'name': 'Bare', 'color': 'tan'},
    2: {'name': 'Water', 'color': 'blue'},
    3: {'name': 'Vegetation', 'color': 'green'}
}

# Create a new training dataset interactively using the map
Map = geemap.Map()
Map.centerObject(point, 12)
Map.addLayer(clipped_image, vis_params, "Landsat RGB")

# Display instructions for collecting training data
print("Creating a new training dataset...")
print("1. Use the map interface to collect training points")
print("2. For each land cover class, collect at least 25 points")
print("3. When finished, export the points to a GEE asset")

# You would use Map.draw_features() here, but since this is interactive,
# let's provide code to create some sample points programmatically
# as an example (you should replace with real data collection)

# Example of programmatically creating some sample points (for demonstration)
def create_sample_points(center_point, radius, n_points, class_value):
    """Create sample points around a center within radius"""
    points_list = []
    # Create a larger buffer to sample from
    buffer = center_point.buffer(radius)
    # Generate random points within the buffer
    random_points = ee.FeatureCollection.randomPoints(
        region=buffer,
        points=n_points,
        seed=123 + class_value
    )
    # Add class label
    labeled_points = random_points.map(lambda f: f.set('label', class_value))
    return labeled_points

# Create some sample points for each class around different locations
# Note: In a real scenario, you would carefully select these locations based on visual interpretation
urban_center = ee.Geometry.Point([-122.419, 37.775])  # Downtown SF
water_center = ee.Geometry.Point([-122.478, 37.808])  # Bay area
veg_center = ee.Geometry.Point([-122.472, 37.737])    # Park area
bare_center = ee.Geometry.Point([-122.499, 37.771])   # Beach/sand area

# Create 25 points for each class
urban_points = create_sample_points(urban_center, 3000, 25, 1)  # Now class 1
bare_points = create_sample_points(bare_center, 3000, 25, 0)    # Now class 0
water_points = create_sample_points(water_center, 3000, 25, 2)  # Unchanged
veg_points = create_sample_points(veg_center, 3000, 25, 3)      # Unchanged

# Merge all points
new_samples = urban_points.merge(bare_points).merge(water_points).merge(veg_points)

# Display the points on the map
Map.addLayer(urban_points, {'color': 'red'}, 'Urban (Class 1)')     # Updated class number
Map.addLayer(bare_points, {'color': 'yellow'}, 'Bare (Class 0)')    # Updated class number
Map.addLayer(water_points, {'color': 'blue'}, 'Water (Class 2)')    # Unchanged
Map.addLayer(veg_points, {'color': 'green'}, 'Vegetation (Class 3)') # Unchanged

Map

# Now use these new points for sampling and classification
sampled_points = normalized_image.sampleRegions(
    collection=new_samples,
    properties=['label'],
    scale=30
)

print("Number of sampled points:", sampled_points.size().getInfo())
print("Sample point example:", sampled_points.first().getInfo())

# Split the data
training_data = sampled_points.randomColumn("random", seed=123)
training_samples = training_data.filter(ee.Filter.lt("random", 0.7))
testing_samples = training_data.filter(ee.Filter.gte("random", 0.7))

# Print dataset sizes
print("Number of training samples:", training_samples.size().getInfo())
print("Number of testing samples:", testing_samples.size().getInfo())
print("Labels in training set:", training_samples.aggregate_array("label").distinct().getInfo())

# Train the classifier
norm_band_names = normalized_image.bandNames()
classifier = ee.Classifier.smileRandomForest(
    numberOfTrees=100,
    minLeafPopulation=1,
    bagFraction=0.7,
    seed=123
).train(
    features=training_samples,
    classProperty='label',
    inputProperties=norm_band_names
)

# Get feature importance
variable_importance = classifier.explain().get('importance')
print("Feature importance scores:")
importance_dict = variable_importance.getInfo()
for band, score in zip(norm_band_names.getInfo(), importance_dict.get('values', [])):
    print(f"{band}: {score}")

Label samples info: {'type': 'Feature', 'geometry': {'type': 'Point', 'coordinates': [26.60200135037303, -54.37590198777617]}, 'id': '00000000000000000000', 'properties': {'Id': 2}}
Number of features: 722
Creating a new training dataset...
1. Use the map interface to collect training points
2. For each land cover class, collect at least 25 points
3. When finished, export the points to a GEE asset
Number of sampled points: 89
Sample point example: {'type': 'Feature', 'geometry': None, 'id': '1_1_1_0_0', 'properties': {'MNDWI_norm': 0.7622059319772315, 'NDBI_norm': 0.43348836402430335, 'NDVI_norm': 0.29321050608800614, 'SR_B1_norm': 0.40661752223968506, 'SR_B2_norm': 0.4020390212535858, 'SR_B3_norm': 0.4171093702316284, 'SR_B4_norm': 0.41184425354003906, 'SR_B5_norm': 0.4343046545982361, 'SR_B6_norm': 0.4255466163158417, 'SR_B7_norm': 0.4406912922859192, 'elevation_norm': 0.31481480598449707, 'label': 1, 'slope_norm': 0.24752778863862224, 'sobel_b2_norm': 0.7205280730705761, 'sobel_b3_n

8 classification and visualization

In [ ]:
# Classify the image
classified_image = normalized_image.classify(classifier)

# Define visualization parameters based on your class labels
# For example, if classes are:
# 0: Urban, 1: Bare, 2: Water, 3: Vegetation
class_vis = {
    'min': 0,
    'max': 3,
    'palette': ['#E63946', '#F1C453', '#457B9D', '#2A9D8F']
}

# Display the classified image
Map = geemap.Map()
Map.centerObject(point, 12)
Map.addLayer(normalized_image, norm_vis_params, "Normalized Image", False)
Map.addLayer(classified_image, class_vis, "Land Cover Classification")
Map.addLayer(label_samples, {}, "Training Points", False)
Map

Map(center=[37.7749, -122.41939999999998], controls=(WidgetControl(options=['position', 'transparent_bg'], wid…

9 accuracy assessment

In [ ]:
# Classify the test samples
test_accuracy = testing_samples.classify(classifier)

# Generate confusion matrix
confusion_matrix = test_accuracy.errorMatrix('label', 'classification')

# Calculate accuracy metrics
overall_accuracy = confusion_matrix.accuracy()
producers_accuracy = confusion_matrix.producersAccuracy()
consumers_accuracy = confusion_matrix.consumersAccuracy()
kappa = confusion_matrix.kappa()

# Print accuracy results
print("Overall Accuracy:", overall_accuracy.getInfo())
print("Kappa Coefficient:", kappa.getInfo())
print("Producer's Accuracy (Recall):", producers_accuracy.getInfo())
print("Consumer's Accuracy (Precision):", consumers_accuracy.getInfo())
print("Confusion Matrix:")
print(confusion_matrix.getInfo())

Overall Accuracy: 0.45161290322580644
Kappa Coefficient: 0.2629370629370629
Producer's Accuracy (Recall): [[0.5], [0.2857142857142857], [0.6], [0.375]]
Consumer's Accuracy (Precision): [[0.42857142857142855, 0.3333333333333333, 0.6666666666666666, 0.3333333333333333]]
Confusion Matrix:
[[3, 0, 0, 3], [1, 2, 2, 2], [2, 1, 6, 1], [1, 3, 1, 3]]


10. comparison

In [ ]:
# Load ESA WorldCover data with the correct asset ID
# The error indicates that 'ESA/WorldCover/v200' is not an Image
# Let's use the correct asset ID for the latest version

try:
    # Try the current version
    esa_worldcover = ee.ImageCollection("ESA/WorldCover/v100").first().clip(region)
    print("Using ESA WorldCover v100")
except:
    try:
        # Try the alternative ID format
        esa_worldcover = ee.ImageCollection("ESA/WorldCover/v200").first().clip(region)
        print("Using ESA WorldCover v200")
    except:
        try:
            # Try another alternative format
            esa_worldcover = ee.Image("ESA/WorldCover/v100/2020").clip(region)
            print("Using ESA WorldCover v100/2020")
        except:
            # If all else fails, use the ESA Land Cover CCI dataset as an alternative
            esa_worldcover = ee.Image("ESA/GLOBCOVER_L4_200901_200912_V2_3").select('landcover').clip(region)
            print("Using ESA GLOBCOVER as alternative")

# Define visualization parameters for ESA WorldCover
# Adjust based on which dataset was successfully loaded
esa_vis = {} # Empty visualization params will use the default visualization

# Display both classifications side by side
Map = geemap.Map()
Map.centerObject(point, 12)
Map.addLayer(classified_image, class_vis, "Your Classification")
Map.addLayer(esa_worldcover, esa_vis, "ESA Land Cover")
Map

Using ESA WorldCover v100


Map(center=[37.7749, -122.41939999999998], controls=(WidgetControl(options=['position', 'transparent_bg'], wid…

## Reflection and Analysis

Overall Model Performance
Accuracy:
The model achieved an overall accuracy of 45%, with a Kappa coefficient of 0.26. While this reflects some ability to distinguish between classes, it also indicates that there is considerable room for improvement.
Limitations Encountered
Limited Sample Size:
With only 89 sampled points (58 for training and 31 for testing), the dataset was small. This scarcity likely contributed to the moderate accuracy and low Kappa, making the model vulnerable to overfitting.

Manual Data Collection:
Creating the training data by hand was time-consuming and may have introduced inconsistencies or biases, especially when ensuring each land cover class was adequately represented.

Class Imbalance:
The dataset showed imbalance across the four classes ([1, 0, 2, 3]). This imbalance is reflected in the differing Producer’s (Recall) and Consumer’s (Precision) accuracies, leading to uneven model performance across classes.

Feature Engineering Impact
Engineered Features Contribution:
The inclusion of spectral indices such as NDVI_norm, MNDWI_norm, and NDBI_norm—along with several spectral reflectance and texture features (e.g., Sobel filters)—played a crucial role. These features were intended to capture the nuanced spectral differences between land cover types.

Most Important Features:
Based on feature importance scores, the most influential features were:

NDVI_norm (Vegetation index)
MNDWI_norm (Water index)
NDBI_norm (Built-up index)
These features align with expectations in remote sensing, where vegetation, water, and built-up areas are often clearly differentiated by their spectral signatures.
Layer Contribution:
Although the full ranking wasn’t provided, the indices mentioned above likely contributed the most to the model’s discrimination power. Texture features (from the Sobel filters) and topographic information (elevation_norm and slope_norm) also added value but perhaps to a lesser degree.

Analysis of Class Confusions
Confused Classes:
The confusion matrix revealed significant misclassifications among specific class pairs:

Classes 1 and 3:
For true class 1, predictions were split evenly between class 1 and class 3.
Classes 0 and 2:
These classes also exhibited notable confusion, with predictions frequently interchanged.
This pattern suggests that the spectral and textural characteristics of these classes are similar enough to cause misclassification.

Implications:
Depending on the application, such confusion could be either acceptable or problematic. For instance, if distinguishing between water and vegetation is critical, misclassifications between these classes would be detrimental. However, if certain classes are less critical, a degree of confusion might be tolerable.

Reflections on Data Collection and Future Improvements
Manual Collection Challenges:
Hand-creating the training data using the map interface proved challenging due to the time investment and potential for human error. This process can lead to non-representative sampling and bias.

Addressing Class Imbalance:
Given the evident class imbalance and its impact on accuracy:

Stratified Sampling: Implementing a more structured approach to ensure equal representation of each class during data collection could help.
Resampling Techniques: Techniques such as oversampling the minority classes or undersampling the majority classes—and even using synthetic data generation methods (e.g., SMOTE)—might further alleviate imbalance issues.
Additional Data and Cross-Validation:
Increasing the overall number of training samples and adopting cross-validation techniques would likely yield more reliable performance estimates and allow for more refined parameter tuning.

Final Thoughts
This exercise has highlighted several key insights:

Data Quality and Quantity:
The importance of a robust, well-balanced training dataset cannot be overstated.
Feature Engineering:
Thoughtful feature engineering—emphasizing indices like NDVI_norm, MNDWI_norm, and NDBI_norm—proved beneficial, though further optimization could improve results.
Class Confusions:
The most confused class pairs (1 & 3 and 0 & 2) underscore the need for improved data collection and feature discrimination strategies, especially in contexts where misclassification could have significant consequences.
In future iterations, addressing these issues with larger, more balanced datasets and advanced sampling techniques could significantly enhance model performance and reliability.